## Example 1: Offline API surface

In [ ]:
import planetary_computer as pc
from planetary_computer.settings import Settings

# Example 1: the OFFLINE surface of the package. Nothing here opens the STAC endpoint.
print(f"planetary_computer version: {pc.version.__version__}")
print(f"planetary_computer location: {pc.__file__}")
assert isinstance(pc.version.__version__, str) and pc.version.__version__, pc.version.__version__

PUBLIC_CALLABLES = (
    "sign", "sign_inplace", "sign_item", "sign_item_collection", "sign_asset", "sign_assets",
    "sign_url", "set_subscription_key", "get_container_client", "get_adlfs_filesystem",
)
missing = [name for name in PUBLIC_CALLABLES if not callable(getattr(pc, name, None))]
assert not missing, f"planetary_computer no longer exposes {missing}"
for name in PUBLIC_CALLABLES:
    print(f"  OK {name}: {type(getattr(pc, name)).__name__}")

# `Settings` lives in `planetary_computer.settings`, not at the top level.
settings = Settings()
print(f"Settings: {type(settings).__name__}")
print(f"  sas_url: {settings.sas_url}")
print(f"  subscription_key: {'[SET]' if settings.subscription_key else None}")
assert settings.sas_url.startswith("https://"), settings.sas_url

attrs = [attr for attr in dir(pc) if not attr.startswith("_")]
print(f"Total public attributes: {len(attrs)}")
assert {"settings", "utils", "version"} <= set(attrs), sorted(attrs)
print("Offline planetary_computer surface check completed")

## Example 2: Catalog access and collections

In [ ]:
import planetary_computer as pc
import pystac_client

# Example 2: Catalog Access and Exploration
print("Example 2: Catalog access and exploration")

STAC_ENDPOINT = "https://planetarycomputer.microsoft.com/api/stac/v1"

# Access the Planetary Computer STAC catalog
print("\n--- STAC Catalog Access ---")

try:
    # Connect to the Planetary Computer STAC API
    catalog = pystac_client.Client.open(STAC_ENDPOINT)
except Exception as e:
    print(f"Catalog connection error: {e}")
    catalog = None

assert catalog is not None, (
    f"the Planetary Computer STAC endpoint ({STAC_ENDPOINT}) did not open (the reason is printed "
    f"above); this cell needs outbound network (run with --with network)"
)
assert catalog.id, f"the catalog opened but has no id: {catalog!r}"
print(f"Catalog connected: {catalog.id}")
print(f"Catalog title: {catalog.title}")
print(f"Catalog description: {catalog.description[:100]}...")

# Explore available collections
if catalog:
    print("\n--- Available Collections ---")
    
    try:
        # Get all collections (one listing, reused below)
        collections = list(catalog.get_collections())
        print(f"Total collections: {len(collections)}")
        assert len(collections) > 0, "the catalog listed no collections"
        assert all(collection.id for collection in collections), "a listed collection has no id"
        
        # Show details of first few collections
        for i, collection in enumerate(collections[:5]):
            print(f"\n{i+1}. {collection.id}")
            print(f"   Title: {collection.title}")
            print(f"   Description: {collection.description[:80]}...")
            
            # Show spatial and temporal extent
            if collection.extent:
                if collection.extent.spatial:
                    bbox = collection.extent.spatial.bboxes[0] if collection.extent.spatial.bboxes else None
                    if bbox:
                        print(f"   Spatial extent: {bbox[:2]} to {bbox[2:]}")
                
                if collection.extent.temporal:
                    intervals = collection.extent.temporal.intervals
                    if intervals and intervals[0]:
                        start, end = intervals[0]
                        print(f"   Temporal extent: {start} to {end}")
        
        # Show collection categories
        collection_ids = [col.id for col in collections[:20]]  # First 20
        print(f"\nSample collection IDs: {collection_ids[:10]}")
        
    except Exception as e:
        print(f"Collection exploration error: {e}")
        raise

# Test specific collection access
print("\n--- Specific Collection Access ---")

try:
    # Try to access a well-known collection
    common_collections = ['sentinel-2-l2a', 'landsat-c2-l2', 'naip', 'modis-21A1D-061']
    
    for collection_id in common_collections[:2]:  # Test first 2
        try:
            collection = catalog.get_collection(collection_id)
            print(f"✓ {collection_id}: {collection.title}")
            assert collection.id == collection_id, (collection.id, collection_id)
            
            # Show some collection metadata
            if hasattr(collection, 'keywords') and collection.keywords:
                print(f"  Keywords: {collection.keywords[:3]}")
            
            if hasattr(collection, 'license'):
                print(f"  License: {collection.license}")
                
        except Exception as col_error:
            print(f"✗ {collection_id}: {col_error}")
            raise
            
except Exception as e:
    print(f"Specific collection access error: {e}")
    raise

# Test collection search
print("\n--- Collection Search ---")

try:
    # Search for Earth observation collections
    search_terms = ['satellite', 'landsat', 'sentinel', 'imagery']
    
    found_collections = []
    for term in search_terms[:2]:  # Limit to 2 search terms
        matching = []
        for col in collections[:20]:  # Search in first 20 collections
            if (term.lower() in col.title.lower() or 
                term.lower() in col.description.lower() or
                term.lower() in col.id.lower()):
                matching.append(col.id)
        
        if matching:
            found_collections.extend(matching[:3])  # Limit results
            print(f"'{term}' matches: {matching[:3]}")
        else:
            print(f"'{term}' matches: None found")
    
    # Remove duplicates
    unique_found = list(dict.fromkeys(found_collections))
    print(f"Unique matching collections: {len(unique_found)}")
    
except Exception as e:
    print(f"Collection search error: {e}")
    raise

print("Catalog access and exploration completed")

## Example 3: Item search and query

In [ ]:
import planetary_computer as pc
import pystac_client
from datetime import datetime, timedelta, timezone
from urllib.parse import parse_qs, urlparse

# Example 3: Item Search and Query
print("Example 3: Item search and query")

STAC_ENDPOINT = "https://planetarycomputer.microsoft.com/api/stac/v1"
CLOUD_COVER_BOUND = 20
AZURE_BLOB_DOMAIN = ".blob.core.windows.net"

def bboxes_intersect(left, right):
    """True when two `[west, south, east, north]` boxes overlap at all."""

    return left[0] <= right[2] and right[0] <= left[2] and left[1] <= right[3] and right[1] <= left[3]

def is_azure_blob_href(href):
    """True for the hrefs `planetary_computer.sign` is supposed to act on (`sas.py` keys on exactly this)."""

    return urlparse(href).netloc.endswith(AZURE_BLOB_DOMAIN)

# Connect to catalog for searching
try:
    catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
    print("Connected to Planetary Computer catalog")
except AssertionError:
    raise
except Exception as e:
    print(f"Catalog connection error: {e}")
    catalog = None

assert catalog is not None, (
    f"the Planetary Computer STAC endpoint ({STAC_ENDPOINT}) did not open (the reason is printed "
    f"above); this cell needs outbound network (run with --with network)"
)
assert catalog.id, f"the catalog opened but has no id: {catalog!r}"
print(f"Catalog id asserted: {catalog.id}")

if catalog:
    print("\n--- Basic Item Search ---")

    try:
        # Define search parameters
        # Use a small area for testing (San Francisco Bay Area)
        bbox = [-122.5, 37.5, -122.0, 38.0]  # West, South, East, North

        end_date = datetime.now(timezone.utc)
        start_date = end_date - timedelta(days=365)  # a year of revisits, so the search cannot be empty

        print(f"Search area: {bbox}")
        print(f"Date range: {start_date.date()} to {end_date.date()}")

        # Search for items in a collection
        collections_to_search = ['sentinel-2-l2a', 'landsat-c2-l2']

        for collection_id in collections_to_search[:1]:  # Test one collection
            try:
                print(f"\nSearching {collection_id}...")

                search = catalog.search(
                    collections=[collection_id],
                    bbox=bbox,
                    datetime=f"{start_date.isoformat()}/{end_date.isoformat()}",
                    limit=5,  # the page size, not a cap
                    max_items=5,  # the cap on the number of items returned
                )

                items = list(search.items())
                assert len(items) <= 5, (
                    f"the cap of 5 did not hold: the search yielded {len(items)} items"
                )
                assert items, (
                    f"{collection_id} returned NO items for bbox {bbox} over "
                    f"{start_date.date()}..{end_date.date()} — a year of revisits over the San "
                    f"Francisco Bay cannot legitimately be empty, so either the search parameters "
                    f"stopped being honoured or the collection moved"
                )
                print(f"Found {len(items)} items")

                # The search parameters (collection, bbox, datetime) were honoured, and the results come back unsigned.
                for item in items:
                    assert item.collection_id == collection_id, (
                        f"`collections=[{collection_id!r}]` was not honoured: {item.id} belongs to "
                        f"{item.collection_id!r}"
                    )
                    assert item.bbox is not None and bboxes_intersect(item.bbox, bbox), (
                        f"`bbox={bbox}` was not honoured: {item.id} has bbox {item.bbox}"
                    )
                    assert item.datetime is not None and start_date <= item.datetime <= end_date, (
                        f"`datetime=` was not honoured: {item.id} is stamped {item.datetime}, outside "
                        f"{start_date}..{end_date}"
                    )
                    for asset_key, asset in item.assets.items():
                        if is_azure_blob_href(asset.href):
                            assert "sig" not in parse_qs(urlparse(asset.href).query), (
                                f"{item.id}/{asset_key} came back from the search already signed. "
                                f"Example 4 tests `planetary_computer.sign` on the assumption that "
                                f"search results are unsigned, and would stop testing it."
                            )
                print(f"  collection, bbox, datetime and unsigned-asset checks hold for all {len(items)} item(s)")

                # Show details of found items
                for i, item in enumerate(items[:3]):  # Show first 3
                    print(f"\nItem {i+1}: {item.id}")
                    print(f"  Collection: {item.collection_id}")
                    print(f"  Datetime: {item.datetime}")
                    print(f"  Geometry type: {item.geometry['type'] if item.geometry else 'None'}")

                    # Show available assets
                    if item.assets:
                        asset_keys = list(item.assets.keys())[:5]  # First 5 assets
                        print(f"  Assets: {asset_keys}")

                    # Show properties
                    if item.properties:
                        prop_keys = list(item.properties.keys())[:5]  # First 5 properties
                        print(f"  Properties: {prop_keys}")

            except AssertionError:
                raise
            except Exception as search_error:
                print(f"Search error for {collection_id}: {search_error}")
                raise AssertionError(
                    f"the {collection_id} item search failed: "
                    f"{type(search_error).__name__}: {search_error}"
                ) from search_error

    except AssertionError:
        raise
    except Exception as e:
        print(f"Item search setup error: {e}")
        raise AssertionError(f"item search setup failed: {type(e).__name__}: {e}") from e

    print("\n--- Search with Filters ---")

    try:
        # Search with cloud cover filter (for optical imagery)
        print("Searching with cloud cover filter...")

        search = catalog.search(
            collections=['sentinel-2-l2a'],
            bbox=bbox,
            datetime=f"{start_date.isoformat()}/{end_date.isoformat()}",
            query={"eo:cloud_cover": {"lt": 20}},  # Less than 20% cloud cover
            limit=3,  # the page size, not a cap
            max_items=3,  # the cap on the number of items returned
        )

        filtered_items = list(search.items())
        assert len(filtered_items) <= 3, (
            f"the cap of 3 did not hold: the search yielded {len(filtered_items)} items"
        )
        print(f"Found {len(filtered_items)} items with <20% cloud cover")

        assert filtered_items, (
            f"the cloud-cover search returned NO items — over a 365-day window on this bbox that is "
            f"not weather, it is `query=` or the collection"
        )
        for item in filtered_items:
            cloud_cover = item.properties.get("eo:cloud_cover")
            assert cloud_cover is not None, (
                f"{item.id} came back from a `query={{'eo:cloud_cover': ...}}` search without an "
                f"`eo:cloud_cover` property at all — the filter cannot have been applied to it"
            )
            assert cloud_cover < CLOUD_COVER_BOUND, (
                f"the STAC query extension was not honoured: {item.id} has "
                f"eo:cloud_cover={cloud_cover}, which is not < {CLOUD_COVER_BOUND}"
            )
        print(f"  cloud-cover filter holds: all {len(filtered_items)} item(s) are below {CLOUD_COVER_BOUND}% cloud")

        for item in filtered_items:
            cloud_cover = item.properties.get('eo:cloud_cover', 'N/A')
            print(f"  {item.id}: {cloud_cover}% cloud cover")

    except AssertionError:
        raise
    except Exception as e:
        print(f"Filtered search error: {e}")
        raise AssertionError(f"the cloud-cover filtered search failed: {type(e).__name__}: {e}") from e

    print("\n--- Asset Information ---")

    try:
        # Examine assets in detail
        if filtered_items:
            sample_item = filtered_items[0]
            print(f"Examining assets for: {sample_item.id}")

            # A STAC item without assets is not a usable result. The `if item.assets:` guards above
            # print nothing when the dict is empty, so this is asserted explicitly.
            assert sample_item.assets, f"{sample_item.id} carries no assets at all"
            assert any(is_azure_blob_href(asset.href) for asset in sample_item.assets.values()), (
                f"{sample_item.id} has {len(sample_item.assets)} asset(s) and NONE on "
                f"`*{AZURE_BLOB_DOMAIN}` — Example 4 signs exactly those, and would sign nothing"
            )

            for asset_key, asset in sample_item.assets.items():
                print(f"\nAsset: {asset_key}")
                print(f"  Title: {asset.title}")
                print(f"  Media type: {asset.media_type}")
                print(f"  URL: {asset.href[:80]}..." if len(asset.href) > 80 else f"  URL: {asset.href}")

                # Show asset roles if available
                if hasattr(asset, 'roles') and asset.roles:
                    print(f"  Roles: {asset.roles}")

                # Show first few assets only
                if list(sample_item.assets.keys()).index(asset_key) >= 3:
                    remaining = len(sample_item.assets) - 4
                    if remaining > 0:
                        print(f"  ... and {remaining} more assets")
                    break

    except AssertionError:
        raise
    except Exception as e:
        print(f"Asset examination error: {e}")
        raise AssertionError(f"asset examination failed: {type(e).__name__}: {e}") from e

print("Item search and query completed")

## Example 4: Asset signing

In [ ]:
import planetary_computer as pc
import pystac
import pystac_client
from datetime import datetime, timezone
from urllib.parse import parse_qs, urlparse

# Example 4: Asset Signing and Authentication
print("Example 4: Asset signing and authentication")

STAC_ENDPOINT = "https://planetarycomputer.microsoft.com/api/stac/v1"
AZURE_BLOB_DOMAIN = ".blob.core.windows.net"
# `sas.py` builds the query as `f"{href}?{token}"` from the SAS service; these are the keys that make
# the URL readable. `sig` is the signature itself, `se` the expiry, `sp` the permissions.
REQUIRED_SAS_KEYS = ("sig", "se", "sp")

def is_azure_blob_href(href):
    """True for the hrefs `sign_url` acts on — it keys on exactly this suffix."""

    return urlparse(href).netloc.endswith(AZURE_BLOB_DOMAIN)

def sas_keys(href):
    """The query keys of `href`, as `sign_url` would see them after `token.sign()`."""

    return set(parse_qs(urlparse(href).query))

def url_without_query(href):
    """`(scheme, netloc, path)`: the part of a URL that signing must not change."""

    parsed = urlparse(href)
    return (parsed.scheme, parsed.netloc, parsed.path)

print("\n--- Signing Contract (no endpoint needed) ---")

# A non-Azure URL is returned unchanged.
PLAIN_URL = "https://example.invalid/data/cog.tif"
assert pc.sign(PLAIN_URL) == PLAIN_URL, (
    f"`sign` rewrote a non-Azure URL. Got {pc.sign(PLAIN_URL)!r}, expected {PLAIN_URL!r}. "
    f"`sign_url` must return any href whose host does not end in {AZURE_BLOB_DOMAIN!r} untouched."
)

# The public-thumbnail account is exempt by name, even though it is a blob host.
AI4E_URL = "https://ai4edatasetspublicassets.blob.core.windows.net/assets/thumbnail.png"
assert pc.sign(AI4E_URL) == AI4E_URL, (
    f"`sign` tried to sign the public `ai4edatasetspublicassets` account. That account has "
    f"no SAS endpoint, so signing it turns a working thumbnail URL into a broken one."
)

# Idempotence: an href that already carries `st`/`se`/`sp` is returned untouched, WITHOUT a second
# round trip to the SAS service. This is what makes re-signing a cached item cheap and safe.
ALREADY_SIGNED_URL = (
    "https://example.blob.core.windows.net/container/blob.tif"
    "?st=2026-09-20T00%3A00%3A00Z&se=2026-09-21T00%3A00%3A00Z&sp=r&sig=NOTAREALSIGNATURE"
)
assert pc.sign(ALREADY_SIGNED_URL) == ALREADY_SIGNED_URL, (
    f"`sign` re-signed an already-signed href instead of returning it. Got "
    f"{pc.sign(ALREADY_SIGNED_URL)!r}."
)

# An unsupported type must raise, not pass through. The `@singledispatch` base raises TypeError;
# a `sign` that had lost its registrations would fall through to the base for EVERY type, so this is
# also the canary for a broken dispatch table.
try:
    pc.sign(1234)
except TypeError as expected_error:
    print(f"  sign(int) refused, as {type(expected_error).__name__}")
else:
    raise AssertionError("`sign(1234)` returned instead of raising TypeError")

# The same at the Asset level: a non-Azure href is unchanged, and sign_asset returns a copy.
offline_asset = pystac.Asset(href=PLAIN_URL)
offline_signed = pc.sign_asset(offline_asset)
assert offline_signed is not offline_asset, (
    "`sign_asset` returned the SAME object — `copy=True` is the default "
    "and must clone, or a caller's catalogue is mutated behind its back"
)
assert offline_signed.href == PLAIN_URL, (
    f"`sign_asset` rewrote a non-Azure href to {offline_signed.href!r}"
)
print("  Offline signing checks passed: plain URL, public account, already-signed URL, unsupported type, copy")

# Connect to catalog
try:
    catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
    print("Connected to Planetary Computer catalog")
except AssertionError:
    raise
except Exception as e:
    print(f"Catalog connection error: {e}")
    catalog = None

assert catalog is not None, (
    f"the Planetary Computer STAC endpoint ({STAC_ENDPOINT}) did not open (the reason is printed "
    f"above); the offline signing checks already passed. This cell needs outbound "
    f"network (run with --with network)"
)

if catalog:
    print("\n--- Finding Items for Signing ---")

    try:
        search = catalog.search(
            collections=['sentinel-2-l2a'],
            bbox=[-122.5, 37.5, -122.0, 38.0],  # San Francisco Bay Area
            limit=1,  # the page size, not a cap
            max_items=1,  # the cap on the number of items returned
        )

        items = list(search.items())
        assert len(items) <= 1, (
            f"the cap of 1 did not hold: the search yielded {len(items)} items"
        )
        assert items, (
            "the unfiltered sentinel-2-l2a search over the San Francisco Bay returned NO items — with "
            "no date filter at all that archive cannot be empty, so the search stopped being honoured"
        )

        if items:
            item = items[0]
            print(f"Found item for signing: {item.id}")
            print(f"Collection: {item.collection_id}")
            print(f"Number of assets: {len(item.assets)}")

            # THE PREMISE OF EVERY CLAIM BELOW: the item arrives UNSIGNED and has something to sign.
            blob_asset_keys = [k for k, a in item.assets.items() if is_azure_blob_href(a.href)]
            assert blob_asset_keys, (
                f"{item.id} has {len(item.assets)} asset(s) and none on `*{AZURE_BLOB_DOMAIN}` — there "
                f"is nothing for `sign` to do, so a broken `sign` would pass every check below"
            )
            for asset_key in blob_asset_keys:
                assert not (sas_keys(item.assets[asset_key].href) & set(REQUIRED_SAS_KEYS)), (
                    f"{item.id}/{asset_key} came back from SEARCH already signed; the before/after "
                    f"comparison this cell is built on would be vacuous"
                )
            print(f"  {len(blob_asset_keys)} of {len(item.assets)} asset(s) are unsigned Azure blobs")

            # Show some asset URLs before signing
            print(f"\n--- Asset URLs Before Signing ---")
            sample_assets = list(item.assets.items())[:3]  # First 3 assets

            for asset_key, asset in sample_assets:
                print(f"{asset_key}:")
                print(f"  URL: {asset.href[:100]}...")
                print(f"  Needs signing: {'blob.core.windows.net' in asset.href}")

        else:
            print("No items found for signing test")
            item = None

    except AssertionError:
        raise
    except Exception as e:
        print(f"Item search error: {e}")
        item = None
        raise AssertionError(f"the item search for signing failed: {type(e).__name__}: {e}") from e

    # Test signing functionality
    if item:
        print("\n--- Asset Signing ---")

        try:
            # Sign the item using planetary_computer.sign
            unsigned_hrefs = {key: asset.href for key, asset in item.assets.items()}
            signed_item = pc.sign(item)

            print(f"Item signed successfully")
            print(f"Original item ID: {item.id}")
            print(f"Signed item ID: {signed_item.id}")

            # `sign` returns a copy. The input item must come back untouched, or a caller that signs one
            # item has silently mutated the search result it still holds.
            assert signed_item is not item, (
                "`pc.sign(item)` returned the SAME object. `copy=True` is the default and "
                "must clone; mutating the caller's item is `sign_inplace`'s job, not `sign`'s."
            )
            assert set(signed_item.assets) == set(item.assets), (
                f"signing changed the asset SET: {sorted(set(item.assets) ^ set(signed_item.assets))}"
            )
            for key, href in unsigned_hrefs.items():
                assert item.assets[key].href == href, (
                    f"signing mutated the ORIGINAL item's {key} href in place"
                )

            # Every asset, not a sample: non-blob hrefs are unchanged; blob hrefs gain a SAS token in the
            # query, keep scheme, host and path, and carry an expiry in the future.
            now = datetime.now(timezone.utc)
            for key, original_href in unsigned_hrefs.items():
                signed_href = signed_item.assets[key].href
                if not is_azure_blob_href(original_href):
                    # Only blob hrefs are signed. `tilejson` / `rendered_preview` live on the API host, not blob
                    # storage; a signer that appended a token to them would corrupt working URLs.
                    assert signed_href == original_href, (
                        f"{key} is not on {AZURE_BLOB_DOMAIN} yet signing changed it from "
                        f"{original_href!r} to {signed_href!r}"
                    )
                    continue
                # A token was added, and only the query changed.
                missing = [k for k in REQUIRED_SAS_KEYS if k not in sas_keys(signed_href)]
                assert not missing, (
                    f"signing {key} produced no {missing} in its query. The href is "
                    f"{signed_href[:160]!r} — an unsigned URL returned from `sign` is a 403 at read time."
                )
                assert url_without_query(signed_href) == url_without_query(original_href), (
                    f"signing {key} rewrote more than the query (scheme, host or path changed).\n"
                    f"  before: {url_without_query(original_href)}\n"
                    f"  after:  {url_without_query(signed_href)}"
                )
                # The token has not expired. `se` comes back percent-decoded from `parse_qs`.
                expiry_text = parse_qs(urlparse(signed_href).query)["se"][0]
                expiry = datetime.fromisoformat(expiry_text.replace("Z", "+00:00"))
                assert expiry > now, (
                    f"the SAS token for {key} expires at {expiry.isoformat()}, which is not "
                    f"after {now.isoformat()} — signing succeeded and produced a dead token"
                )
            print(f"  Signed hrefs checked for all {len(unsigned_hrefs)} asset(s)")

            # Re-signing an already-signed item is a no-op, so a pipeline that
            # signs defensively does not burn a SAS request per pass.
            twice_signed = pc.sign(signed_item)
            for key in signed_item.assets:
                assert twice_signed.assets[key].href == signed_item.assets[key].href, (
                    f"re-signing the signed item changed {key}"
                )
            print("  Re-signing the signed item is a no-op")

            # Compare URLs before and after signing
            print(f"\n--- Asset URLs After Signing ---")

            for asset_key in sample_assets[:3]:  # Same assets as before
                asset_key_name = asset_key[0]

                if asset_key_name in signed_item.assets:
                    original_asset = item.assets[asset_key_name]
                    signed_asset = signed_item.assets[asset_key_name]

                    print(f"{asset_key_name}:")
                    print(f"  Original: {original_asset.href[:80]}...")
                    print(f"  Signed: {signed_asset.href[:80]}...")

                    # Check if URL was modified
                    urls_different = original_asset.href != signed_asset.href
                    print(f"  URL modified: {urls_different}")

                    if urls_different:
                        # Check for SAS token parameters
                        has_sas = any(param in signed_asset.href.lower()
                                    for param in ['sig=', 'se=', 'sp=', 'sv='])
                        print(f"  Has SAS token: {has_sas}")

        except AssertionError:
            raise
        except Exception as signing_error:
            print(f"Signing error: {signing_error}")
            raise AssertionError(
                f"signing the item failed: {type(signing_error).__name__}: {signing_error}"
            ) from signing_error

    print("\n--- In-Place Signing Test ---")

    try:
        if item:
            # Create a copy for in-place signing
            import copy
            item_copy = copy.deepcopy(item)

            # Test in-place signing
            returned = pc.sign_inplace(item_copy)
            print("In-place signing completed")

            # `sign_inplace` is `sign(obj, copy=False)`: it returns the SAME object and mutates it.
            # A `sign_inplace` that cloned instead would leave the caller's item unsigned, which is
            # exactly the difference the `copy` flag expresses.
            assert returned is item_copy, (
                "`sign_inplace` returned a DIFFERENT object. Callers rely on it mutating the "
                "item they passed; a clone means their own item is still unsigned."
            )
            signed_in_place = [
                key for key, asset in item_copy.assets.items()
                if is_azure_blob_href(asset.href) and "sig" in sas_keys(asset.href)
            ]
            assert len(signed_in_place) == len(blob_asset_keys), (
                f"`sign_inplace` signed {len(signed_in_place)} of {len(blob_asset_keys)} "
                f"Azure blob asset(s) in place"
            )
            # ... and the ORIGINAL, which was only deep-copied, is still untouched.
            for key in blob_asset_keys:
                assert "sig" not in sas_keys(item.assets[key].href), (
                    f"`sign_inplace` on a deepcopy reached back into the original item's {key}"
                )
            print(f"  sign_inplace returned the same object and signed {len(signed_in_place)} asset(s) in place")

            # Verify the original item was modified
            sample_asset_key = list(item.assets.keys())[0]
            original_url = item.assets[sample_asset_key].href
            modified_url = item_copy.assets[sample_asset_key].href

            print(f"Original URL: {original_url[:80]}...")
            print(f"Modified URL: {modified_url[:80]}...")
            print(f"URLs different: {original_url != modified_url}")

    except AssertionError:
        raise
    except Exception as e:
        print(f"In-place signing error: {e}")
        raise AssertionError(f"in-place signing failed: {type(e).__name__}: {e}") from e

    # Test sign_item function
    print("\n--- Individual Item Signing ---")

    try:
        if item:
            # Test signing individual items
            individual_signed = pc.sign_item(item)

            print("Individual item signing completed")
            print(f"Type: {type(individual_signed)}")
            print(f"Item ID: {individual_signed.id}")

            # Check if assets were signed
            signed_assets = len([a for a in individual_signed.assets.values()
                               if 'sig=' in a.href.lower()])
            total_assets = len(individual_signed.assets)

            print(f"Signed assets: {signed_assets}/{total_assets}")

            # `sign_item` is the registered implementation `sign` dispatches an Item to, so the two must
            # agree.
            assert isinstance(individual_signed, pystac.Item), (
                f"`sign_item` returned a {type(individual_signed).__name__}, not a pystac Item"
            )
            assert signed_assets == len(blob_asset_keys), (
                f"`sign_item` signed {signed_assets} of the {len(blob_asset_keys)} Azure blob asset(s) "
                f"on this item — `sign` and `sign_item` are the same code path and must agree"
            )
            for key in blob_asset_keys:
                # The SIGNATURES differ between calls (each is a fresh token), so compare the part that
                # must not move: scheme, host and path.
                assert url_without_query(individual_signed.assets[key].href) == url_without_query(
                    signed_item.assets[key].href
                ), f"`sign_item` and `sign` disagree on the base URL of {key}"

    except AssertionError:
        raise
    except Exception as e:
        print(f"Individual signing error: {e}")
        raise AssertionError(f"individual item signing failed: {type(e).__name__}: {e}") from e

print("Asset signing and authentication completed")

## Example 5: End-to-end search and signing workflow

In [ ]:
import planetary_computer as pc
import pystac
import pystac_client
from urllib.parse import parse_qs, urlparse

# Example 5: End-to-end search and signing workflow
print("Example 5: End-to-end search and signing workflow")

STAPLE_COLLECTION_IDS = ("sentinel-2-l2a", "landsat-c2-l2", "naip")
AZURE_BLOB_DOMAIN = ".blob.core.windows.net"

def is_azure_blob_href(href):
    """True for the hrefs `planetary_computer.sign` acts on — `sas.py` keys on exactly this suffix."""

    return urlparse(href).netloc.endswith(AZURE_BLOB_DOMAIN)

def is_signed_href(href):
    """True when `href` carries a SAS signature, read from the QUERY rather than by substring."""

    return "sig" in parse_qs(urlparse(href).query)

def azure_asset_keys(item):
    """The asset keys of `item` that signing is supposed to act on."""

    return [key for key, asset in item.assets.items() if is_azure_blob_href(asset.href)]

# Comprehensive workflow demonstration
print("\n--- End-to-End Workflow ---")

try:
    # Step 1: Connect to catalog
    catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
    print("✓ Connected to Planetary Computer catalog")

    # Step 2: Explore data availability
    print("\n--- Data Availability Assessment ---")

    # Get collection statistics
    collections = list(catalog.get_collections())
    collection_stats = {
        'total_collections': len(collections),
        'earth_observation': 0,
        'weather_climate': 0,
        'other': 0
    }

    # This is the Planetary Computer catalogue, and the ids the rest of the cell hard-codes are
    # really in it. An empty `list(catalog.get_collections())` would otherwise pass through every
    # `for collection in collections[:20]` below and print a table of zeroes.
    collection_ids = {collection.id for collection in collections}
    missing_staples = [cid for cid in STAPLE_COLLECTION_IDS if cid not in collection_ids]
    assert not missing_staples, (
        f"the catalogue does not carry {missing_staples} — this cell, Example 3 and Example 4 all "
        f"hard-code those ids, so they would each be searching something that does not exist. "
        f"{len(collections)} collection(s) were returned."
    )
    for collection_id in STAPLE_COLLECTION_IDS:
        assert catalog.get_collection(collection_id).id == collection_id, (
            f"`get_collection({collection_id!r})` returned a different collection"
        )
    print(f"✓ {len(collections)} collections; {", ".join(STAPLE_COLLECTION_IDS)} present and fetchable by id")

    # Categorize collections
    earth_obs_keywords = ['satellite', 'imagery', 'sentinel', 'landsat', 'optical', 'radar']
    weather_keywords = ['weather', 'climate', 'temperature', 'precipitation', 'atmospheric']

    for collection in collections[:20]:  # Analyze first 20 collections
        title_desc = (collection.title + ' ' + collection.description).lower()

        if any(keyword in title_desc for keyword in earth_obs_keywords):
            collection_stats['earth_observation'] += 1
        elif any(keyword in title_desc for keyword in weather_keywords):
            collection_stats['weather_climate'] += 1
        else:
            collection_stats['other'] += 1

    print(f"Collection statistics (sample of 20):")
    for category, count in collection_stats.items():
        if category != 'total_collections':
            print(f"  {category.replace('_', ' ').title()}: {count}")

    # Step 3: Multi-collection search
    print("\n--- Multi-Collection Search ---")

    # Search across multiple collections for comparison
    collections_to_compare = ['sentinel-2-l2a', 'landsat-c2-l2']
    bbox = [-122.0, 37.0, -121.5, 37.5]  # Smaller area for faster search

    search_results = {}
    for collection_id in collections_to_compare:
        try:
            search = catalog.search(
                collections=[collection_id],
                bbox=bbox,
                limit=3,  # the page size, not a cap
                max_items=3,  # the cap on the number of items returned
            )

            items = list(search.items())
            assert len(items) <= 3, (
                f"the cap of 3 did not hold: the search yielded {len(items)} items"
            )
            search_results[collection_id] = items
            print(f"  {collection_id}: {len(items)} items found")

        except AssertionError:
            raise
        except Exception as e:
            print(f"  {collection_id}: Search error - {e}")
            search_results[collection_id] = []
            raise AssertionError(
                f"the {collection_id} search failed: {type(e).__name__}: {e}"
            ) from e

    # Both collections answered, and answered with their own items.
    for collection_id, found in search_results.items():
        assert found, (
            f"{collection_id} returned no items for bbox {bbox} with no date filter at all — that "
            f"archive covers this area continuously, so this is the search, not the data"
        )
        for found_item in found:
            assert found_item.collection_id == collection_id, (
                f"`collections=[{collection_id!r}]` was not honoured: {found_item.id} belongs to "
                f"{found_item.collection_id!r}"
            )
    print(f"✓ Both collections returned their own items")

    # Step 4: Asset comparison across collections
    print("\n--- Asset Analysis ---")

    for collection_id, items in search_results.items():
        if items:
            sample_item = items[0]
            asset_types = {}

            for asset_key, asset in sample_item.assets.items():
                media_type = getattr(asset, 'media_type', 'unknown')
                asset_types[media_type] = asset_types.get(media_type, 0) + 1

            print(f"{collection_id} asset types:")
            for media_type, count in asset_types.items():
                print(f"  {media_type}: {count} assets")

    # Step 5: Demonstrate signing workflow
    print("\n--- Signing Workflow ---")

    # Find items that need signing
    items_needing_signing = []
    for collection_id, items in search_results.items():
        for item in items:
            # Check if any assets need signing (contain Azure blob storage URLs)
            needs_signing = any('blob.core.windows.net' in asset.href
                              for asset in item.assets.values())
            if needs_signing:
                items_needing_signing.append((collection_id, item))
                break  # One per collection is enough

    print(f"Items needing signing: {len(items_needing_signing)}")

    # An empty candidate list is a failure, not a quiet skip. Everything from here to the end of
    # the workflow iterates over this list; at zero it would exercise `planetary_computer` not at all.
    assert items_needing_signing, (
        f"no item out of {sum(len(v) for v in search_results.values())} carries an Azure blob asset, "
        f"so this workflow would sign nothing and every check below it would be vacuous"
    )

    # Sign items and demonstrate access
    signed_items = []
    expected_signed_assets = 0
    for collection_id, item in items_needing_signing[:2]:  # Limit to 2 items
        try:
            unsigned_azure_keys = azure_asset_keys(item)
            signed_item = pc.sign(item)
            signed_items.append((collection_id, signed_item))

            # Count signed assets
            signed_asset_count = sum(1 for asset in signed_item.assets.values()
                                   if any(param in asset.href.lower()
                                         for param in ['sig=', 'se=', 'sp=']))

            print(f"  {collection_id}: {signed_asset_count}/{len(signed_item.assets)} assets signed")

            # Coverage, by set not by count, and derived from the query rather than a substring.
            # `signed_asset_count` above is a printed counter that matches `'sig='` anywhere in the
            # href; the comparison below parses the query, so it cannot be fooled by an asset whose URL
            # merely contains those characters.
            actually_signed = {
                key for key, asset in signed_item.assets.items() if is_signed_href(asset.href)
            }
            assert actually_signed == set(unsigned_azure_keys), (
                f"{collection_id}: `pc.sign` signed {sorted(actually_signed)} but the assets on "
                f"{AZURE_BLOB_DOMAIN} are {sorted(unsigned_azure_keys)} — "
                f"left unsigned: {sorted(set(unsigned_azure_keys) - actually_signed)}; "
                f"signed but not a blob: {sorted(actually_signed - set(unsigned_azure_keys))}"
            )
            expected_signed_assets += len(unsigned_azure_keys)

        except AssertionError:
            raise
        except Exception as e:
            print(f"  {collection_id}: Signing error - {e}")
            raise AssertionError(
                f"signing a {collection_id} item failed: {type(e).__name__}: {e}"
            ) from e

    # Step 6: Data access preparation
    print("\n--- Data Access Preparation ---")

    access_summary = {
        'total_signed_items': len(signed_items),
        'access_ready_assets': 0,
        'asset_formats': set()
    }

    for collection_id, signed_item in signed_items:
        for asset_key, asset in signed_item.assets.items():
            if 'sig=' in asset.href.lower():  # Has access token
                access_summary['access_ready_assets'] += 1
                if hasattr(asset, 'media_type'):
                    access_summary['asset_formats'].add(asset.media_type)

    print(f"Access summary:")
    print(f"  Ready-to-access assets: {access_summary['access_ready_assets']}")
    print(f"  Available formats: {list(access_summary['asset_formats'])[:5]}")  # First 5

    # The summary must agree with what signing actually did.
    assert access_summary['access_ready_assets'] == expected_signed_assets, (
        f"the access summary counts {access_summary['access_ready_assets']} ready asset(s) but "
        f"{expected_signed_assets} were signed — the summary and the signing step disagree"
    )
    assert access_summary['access_ready_assets'] > 0, (
        "nothing in this workflow ended up readable: zero assets carry a SAS signature"
    )
    print(f"✓ {expected_signed_assets} asset(s) signed and counted")

    # `sign_item_collection`, reached through `sign`'s dispatch on `pystac.ItemCollection`.
    # Example 1 only checks that the name exists on the module; this calls it. It is the entry point a
    # caller uses after `catalog.search(...)`, which is the common case.
    collection_inputs = [item for _, item in items_needing_signing[:2]]
    unsigned_by_item = [azure_asset_keys(item) for item in collection_inputs]
    signed_collection = pc.sign(pystac.ItemCollection(collection_inputs))
    assert isinstance(signed_collection, pystac.ItemCollection), (
        f"`sign(ItemCollection)` returned a {type(signed_collection).__name__}"
    )
    assert len(signed_collection.items) == len(collection_inputs), (
        f"`sign(ItemCollection)` returned {len(signed_collection.items)} of "
        f"{len(collection_inputs)} item(s)"
    )
    for position, signed_collection_item in enumerate(signed_collection.items):
        for key in unsigned_by_item[position]:
            assert is_signed_href(signed_collection_item.assets[key].href), (
                f"`sign(ItemCollection)` left {signed_collection_item.id}/{key} unsigned"
            )
        assert not any(
            is_signed_href(collection_inputs[position].assets[key].href)
            for key in unsigned_by_item[position]
        ), (
            f"`sign(ItemCollection)` mutated the INPUT collection's "
            f"{collection_inputs[position].id} — the default is `copy=True`"
        )
    print(f"✓ sign(ItemCollection) signed {len(signed_collection.items)} item(s), input untouched")

except AssertionError:
    raise
except Exception as e:
    print(f"Workflow error: {e}")
    raise AssertionError(
        f"planetary_computer example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

print("End-to-end workflow completed")